# Model A + C — OpenLane V1 Data Audit

## 목적

OpenLane V1을 교통사고 과실비율 AI의 Model A + Model C 주 데이터셋으로 사용할 수 있는지 검증한다.

이번 Audit에서는 모델 학습이나 전처리를 바로 시작하지 않는다.

먼저 OpenLane V1의 실제 데이터를 직접 확인하여 다음을 검증한다.

- 이미지와 Lane annotation의 정확한 대응 관계
- 이미지와 CIPO annotation의 정확한 대응 관계
- Scene annotation 구조
- 2D lane (`uv`)의 실제 annotation 품질
- 3D lane (`xyz`)의 실제 annotation 품질
- lane continuity 및 fragmentation 정도
- lane category
- lane visibility
- ego-relative lane attribute
- lane track_id
- CIPO bounding box
- CIPO object type
- CIPO importance level
- CIPO trackid
- 2D / 3D / BEV 시각화 가능 여부
- 사고 분석에 필요한 도로 구조 및 객체 정보를 충분히 제공하는지

최종 목표:

1. 소규모 sanity check
2. 약 10,000개의 matched frame을 이용한 대규모 Data Audit
3. Audit 결과를 바탕으로 Model A + C architecture 확정
4. 이후 OpenLane V1 preprocessing 설계

# 환경 / 경로 / Lane↔CIPO 매칭 기본 구조

In [6]:
from pathlib import Path
import json
import random

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image


# ============================================================
# OpenLane V1 Data Paths
# ============================================================

RAW_ROOT = Path(r"C:\accident_ai\lane_data\open_lane_data\raw")

LANE_ROOT = RAW_ROOT / "lane3d_1000"
CIPO_ROOT = RAW_ROOT / "cipo"
SCENE_JSON_PATH = RAW_ROOT / "scene" / "scene.json"

SPLIT = "training"
SEED = 42

random.seed(SEED)


# ============================================================
# Path Check
# ============================================================

assert RAW_ROOT.exists(), f"RAW_ROOT not found: {RAW_ROOT}"
assert LANE_ROOT.exists(), f"LANE_ROOT not found: {LANE_ROOT}"
assert CIPO_ROOT.exists(), f"CIPO_ROOT not found: {CIPO_ROOT}"
assert SCENE_JSON_PATH.exists(), f"Scene JSON not found: {SCENE_JSON_PATH}"

assert (LANE_ROOT / SPLIT).exists(), (
    f"Lane split not found: {LANE_ROOT / SPLIT}"
)

assert (CIPO_ROOT / SPLIT).exists(), (
    f"CIPO split not found: {CIPO_ROOT / SPLIT}"
)


# ============================================================
# Helper Functions
# ============================================================

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def valid_json_files(root_dir: Path):
    """
    실제 JSON 파일만 가져온다.
    macOS metadata (__MACOSX / ._*)는 제외한다.
    """
    files = []

    for path in root_dir.rglob("*.json"):

        if "__MACOSX" in path.parts:
            continue

        if path.name.startswith("._"):
            continue

        files.append(path)

    return sorted(files)


def list_segment_names(base_dir: Path):
    """
    split 폴더 안의 실제 segment 이름 목록
    """
    return sorted(
        path.name
        for path in base_dir.iterdir()
        if path.is_dir()
        and path.name != "__MACOSX"
        and not path.name.startswith("._")
    )


def get_frame_id(path: Path, source: str):
    """
    OpenLane V1의 서로 다른 파일명 규칙을
    동일한 frame_id로 통일한다.

    Lane
        123456.json
        -> 123456

    CIPO
        123456.jpg.json
        -> 123456
    """

    if source == "lane":
        return path.stem

    if source == "cipo":

        file_name = path.name

        if file_name.endswith(".jpg.json"):
            return file_name[:-len(".jpg.json")]

        # 예상하지 못한 naming이 있을 경우
        return path.stem

    raise ValueError(f"Unknown source: {source}")


def build_frame_map(segment_dir: Path, source: str):
    """
    frame_id -> JSON path
    """
    mapping = {}

    for path in valid_json_files(segment_dir):

        frame_id = get_frame_id(
            path=path,
            source=source,
        )

        if frame_id in mapping:
            raise ValueError(
                f"Duplicate frame_id detected: {frame_id}\n"
                f"Existing: {mapping[frame_id]}\n"
                f"New     : {path}"
            )

        mapping[frame_id] = path

    return mapping


# ============================================================
# Lane ↔ CIPO Segment Matching
# ============================================================

lane_segments = list_segment_names(
    LANE_ROOT / SPLIT
)

cipo_segments = list_segment_names(
    CIPO_ROOT / SPLIT
)

common_segments = sorted(
    set(lane_segments) & set(cipo_segments)
)


print("RAW_ROOT :", RAW_ROOT)
print("LANE_ROOT:", LANE_ROOT)
print("CIPO_ROOT:", CIPO_ROOT)
print("SPLIT    :", SPLIT)

print()

print(f"Lane {SPLIT} segments : {len(lane_segments)}")
print(f"CIPO {SPLIT} segments : {len(cipo_segments)}")
print(f"Common segments       : {len(common_segments)}")


# ============================================================
# Select One Segment
# ============================================================

SEGMENT_INDEX = 0

segment_name = common_segments[SEGMENT_INDEX]

lane_segment_dir = (
    LANE_ROOT
    / SPLIT
    / segment_name
)

cipo_segment_dir = (
    CIPO_ROOT
    / SPLIT
    / segment_name
)


# ============================================================
# Lane ↔ CIPO Frame Matching
# ============================================================

lane_frame_map = build_frame_map(
    lane_segment_dir,
    source="lane",
)

cipo_frame_map = build_frame_map(
    cipo_segment_dir,
    source="cipo",
)

common_frame_ids = sorted(
    set(lane_frame_map) & set(cipo_frame_map)
)


print("\nSelected segment:")
print(segment_name)

print()

print("Lane frame count   :", len(lane_frame_map))
print("CIPO frame count   :", len(cipo_frame_map))
print("Common frame count :", len(common_frame_ids))

print("\nFirst 10 common frame ids:")

for frame_id in common_frame_ids[:10]:
    print(" -", frame_id)


assert len(common_frame_ids) > 0, (
    "Lane과 CIPO의 공통 frame을 찾지 못했습니다."
)

RAW_ROOT : C:\accident_ai\lane_data\open_lane_data\raw
LANE_ROOT: C:\accident_ai\lane_data\open_lane_data\raw\lane3d_1000
CIPO_ROOT: C:\accident_ai\lane_data\open_lane_data\raw\cipo
SPLIT    : training

Lane training segments : 798
CIPO training segments : 798
Common segments       : 798

Selected segment:
segment-10017090168044687777_6380_000_6400_000_with_camera_labels

Lane frame count   : 198
CIPO frame count   : 198
Common frame count : 198

First 10 common frame ids:
 - 155008346734637000
 - 155008346744616300
 - 155008346754599000
 - 155008346764582300
 - 155008346774566500
 - 155008346784553900
 - 155008346794541500
 - 155008346804532700
 - 155008346814523300
 - 155008346824518000


# 동일한 실제 Frame의 Lane JSON + CIPO JSON 열기

In [7]:
# ============================================================
# Cell 3 — Load One Matched Frame
# ============================================================

FRAME_INDEX = 0

frame_id = common_frame_ids[FRAME_INDEX]

lane_json_path = lane_frame_map[frame_id]
cipo_json_path = cipo_frame_map[frame_id]

lane_json = load_json(lane_json_path)
cipo_json = load_json(cipo_json_path)


print("Selected segment :", segment_name)
print("Selected frame   :", frame_id)

print("\n[Lane JSON Path]")
print(lane_json_path)

print("\n[CIPO JSON Path]")
print(cipo_json_path)

print("\n[Lane JSON Top-Level Keys]")
print(list(lane_json.keys()))

print("\n[CIPO JSON Top-Level Keys]")
print(list(cipo_json.keys()))

Selected segment : segment-10017090168044687777_6380_000_6400_000_with_camera_labels
Selected frame   : 155008346734637000

[Lane JSON Path]
C:\accident_ai\lane_data\open_lane_data\raw\lane3d_1000\training\segment-10017090168044687777_6380_000_6400_000_with_camera_labels\155008346734637000.json

[CIPO JSON Path]
C:\accident_ai\lane_data\open_lane_data\raw\cipo\training\segment-10017090168044687777_6380_000_6400_000_with_camera_labels\155008346734637000.jpg.json

[Lane JSON Top-Level Keys]
['extrinsic', 'intrinsic', 'lane_lines', 'file_path', 'pose']

[CIPO JSON Top-Level Keys]
['raw_file_path', 'result']


# 실제 lane_lines 구조 확인

In [8]:
# ============================================================
# Cell 4 — Inspect Actual Lane Annotation Structure
# ============================================================

lane_lines = lane_json["lane_lines"]

print("Lane count :", len(lane_lines))


if len(lane_lines) > 0:

    first_lane = lane_lines[0]

    print("\n[First Lane Keys]")
    print(list(first_lane.keys()))


    print("\n[First Lane Basic Values]")

    for key, value in first_lane.items():

        if key in ["uv", "xyz", "visibility"]:
            array = np.asarray(value)

            print(
                f"{key:12s} : "
                f"type={type(value).__name__}, "
                f"shape={array.shape}"
            )

        else:
            print(
                f"{key:12s} : {value}"
            )


print("\n[Frame-Level Geometry Information]")

print(
    "intrinsic shape :",
    np.asarray(lane_json["intrinsic"]).shape
)

print(
    "extrinsic shape :",
    np.asarray(lane_json["extrinsic"]).shape
)

print(
    "pose shape      :",
    np.asarray(lane_json["pose"]).shape
)

print(
    "file_path       :",
    lane_json["file_path"]
)

Lane count : 2

[First Lane Keys]
['category', 'visibility', 'uv', 'xyz', 'attribute', 'track_id']

[First Lane Basic Values]
category     : 21
visibility   : type=list, shape=(38,)
uv           : type=list, shape=(2, 44)
xyz          : type=list, shape=(3, 38)
attribute    : 0
track_id     : 1

[Frame-Level Geometry Information]
intrinsic shape : (3, 3)
extrinsic shape : (4, 4)
pose shape      : (4, 4)
file_path       : training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels/155008346734637000.jpg
